# S&P 500 協整法配對交易 (Cointegration Method)
## 基於 Johansen 協整檢定的嚴格實作

本 Notebook 回測協整法策略，完全遵守以下規則：

**一、 策略核心架構與時間視窗**
- 滾動窗口（Rolling Window）機制，更新頻率 21 個交易日（約一個月）。
- 形成期 (Formation Period)：252 個交易日（約一年）。
- 交易期 (Trading Period)：126 個交易日（約半年）。
- 隨時維持 6 個重疊組合（$126 \div 21 = 6$），各佔資金 $1/6$。
- 使用 S&P 500 成分股（含息報價）。

**二、 第一階段：配對篩選**
- 形成期首日價格正規化為 1。
- 進行 Johansen 協整檢定，依 Trace Statistics 最高分選前 20 對。
- 參數估計：$P_{1,t} - \beta P_{2,t} = \mu + \epsilon_t$，取得 $\beta$、$\mu$、$\sigma$。

**三、 第二階段：交易規則**
- 交易期首日將價格再次縮放為 1。
- 殘差 $Spread > \mu + 2\sigma$ $\rightarrow$ 賣空 1，買入 2。
- 殘差 $Spread < \mu - 2\sigma$ $\rightarrow$ 買入 1，賣空 2。
- 依 $\beta$ 比例建倉。
- 出場點：回歸均值 $\mu$ 收斂，或第 126 天強制平倉。

**四、 資金補填 (Market Fill)**
- 每個子組合等權重分配前 20 對。
- 若具協整關係少於 10 對，剩餘未使用資金直接做多 SPY (S&P 500)。

**五、 成本基準**
- 單邊交易與滑價：0.3% (30 bps)。
- 放空借券：年化 1%。


In [1]:
# 套件安裝與導入
import subprocess, sys
for pkg in ['statsmodels', 'plotly', 'kaleido', 'yfinance', 'joblib']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

import warnings
warnings.filterwarnings('ignore')
import sqlite3, numpy as np, pandas as pd
import logging
from itertools import combinations
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
from IPython.display import display, HTML
import yfinance as yf
import time
import math
import os

pd.set_option('display.float_format', '{:.4f}'.format)
print('✓ 套件導入完成')
import statsmodels.api as sm
from statsmodels.tsa.vector_ar.vecm import coint_johansen


✓ 套件導入完成


In [2]:
# ========== 階段 1：數據預處理與正規化 ==========
# === 全局參數設置 ===
FAST_TEST_MODE = False

if FAST_TEST_MODE:
    print("【快速測試模式】: 僅回測 2019-2022 年間之科技板塊。")
    # 縮短時間：涵蓋 2020 疫情崩盤與 2022 升息的壓力測試區間
    START_DATE = '2019-01-01'
    END_DATE = '2022-12-31'
    # 縮小股票池：僅測試單一板塊，運算量大幅減少
    TARGET_SECTOR = 'Information Technology'  
else:
    print("【完整回測模式】: 回測 2000-2025 年間之全市場 S&P 500。")
    # 論文要求的全樣本時間
    START_DATE = '2000-01-01'
    END_DATE = '2025-12-31'
    # 測試全市場 (設為 None 代表不限制單一產業)
    TARGET_SECTOR = None  

# 資料庫配置
DB_PATH = r'..\data\SP500.db'
IMPUTED_SECTOR_PATH = r'..\data\imputed_sectors.csv'
USE_DYNAMIC_SECTORS = True  # 是否使用動態產業補齊

# 視窗參數 
FORMATION_WINDOW = 252    # 約 1 年
TRADING_WINDOW = 126      # 約 6 個月
ROLLING_WINDOW = 21       # 約 1 個月 (梯隊步長)
NUM_SUBPORTFOLIOS = math.ceil(TRADING_WINDOW / ROLLING_WINDOW)
MIN_HISTORY_DAYS = 200    # 最少歷史資料

# 配對篩選參數
TOP_N_PAIRS = 20          # 每個視窗選出前 N 組配對
SECTOR_NEUTRAL = True     # 行業中性化
MIN_VALID_PAIRS = 10    # 觸發 Market Fill 門檻

# 交易參數
Z_ENTRY = 2.0             # 進場標準差閾值（SSD 方法中改用 2×σ_formation）
Z_EXIT = 0.0              # 平倉標準差閾值
TRANSACTION_COST = 0.0030 # 雙邊交易成本 (0.29%)
MAX_LOSS_PCT = 0          # 停損 (0 = 無停損)


# 資金配置
INITIAL_CAPITAL = 10000  # 初始本金
SHORT_FEE_ANNUAL = 0.01   # 放空年化費率

# 確保路徑存在
os.makedirs(os.path.dirname(IMPUTED_SECTOR_PATH), exist_ok=True)

print(f"""
【回測參數設置】
時間期間: {START_DATE} ~ {END_DATE}
形成期: {FORMATION_WINDOW} 天（約 1 年）
交易期: {TRADING_WINDOW} 天（約 6 個月）
步長: {ROLLING_WINDOW} 天（約 1 個月）
配對數: {TOP_N_PAIRS}
行業中性: {SECTOR_NEUTRAL}
初始資金: ${INITIAL_CAPITAL:,.0f}
交易成本: {TRANSACTION_COST*100:.2f}%
""")

【完整回測模式】: 回測 2000-2025 年間之全市場 S&P 500。

【回測參數設置】
時間期間: 2000-01-01 ~ 2025-12-31
形成期: 252 天（約 1 年）
交易期: 126 天（約 6 個月）
步長: 21 天（約 1 個月）
配對數: 20
行業中性: True
初始資金: $10,000
交易成本: 0.30%



In [3]:
# === 數據加載函數 ===
def load_data_from_db(db_path, start_date, end_date):
    """從 SQLite 資料庫加載股票價格和行業分類"""
    conn = sqlite3.connect(db_path)
    
    # 嘗試多個可能的表名
    price_queries = [
        (f"SELECT date, ticker, adj_close AS close FROM daily_prices "
         f"WHERE date BETWEEN '{start_date}' AND '{end_date}' ORDER BY date"),
        (f"SELECT date, ticker, close FROM stock_prices "
         f"WHERE date BETWEEN '{start_date}' AND '{end_date}' ORDER BY date"),
    ]
    
    prices_df = None
    for q in price_queries:
        try:
            prices_df = pd.read_sql_query(q, conn, parse_dates=['date'])
            if len(prices_df) > 0:
                print(f'✓ 加載價格數據：{len(prices_df):,} 筆記錄')
                break
        except Exception as e:
            print(f"❌ 讀取資料庫時發生錯誤: {e}")
            continue
    
    if prices_df is None or len(prices_df) == 0:
        raise RuntimeError("無法從資料庫加載價格數據")
    
    # 加載行業分類
    sector_queries = [
        "SELECT ticker, sector FROM tickers",
        "SELECT ticker, sector FROM sp500_components GROUP BY ticker",
    ]
    
    sector_df = None
    for q in sector_queries:
        try:
            sector_df = pd.read_sql_query(q, conn)
            if len(sector_df) > 0:
                print(f'✓ 加載行業數據：{len(sector_df):,} 檔股票')
                break
        except Exception:
            continue
    
    conn.close()
    
    if sector_df is None or len(sector_df) == 0:
        print('⚠️  未找到行業表，使用 Unknown 代替')
        sector_df = pd.DataFrame({'ticker': prices_df['ticker'].unique(), 'sector': 'Unknown'})
    
    return prices_df, sector_df

def fix_unknown_sectors(sector_df, use_dynamic=USE_DYNAMIC_SECTORS, save_path=r'data\imputed_sectors.csv'):
    """具備本機快取與全域開關控制的產業補齊模組"""
    # 1. 開關判斷：如果不使用動態補齊，直接原封不動回傳
    if not use_dynamic:
        print("不使用動態產業補齊。")
        return sector_df

    # 確保儲存的目錄 (data\) 存在
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    
    # 2. 快取讀取：如果已經抓過並存檔，直接載入
    if os.path.exists(save_path):
        print(f"從本機快取載入已補齊的產業分類: {save_path}")
        cached_df = pd.read_csv(save_path)
        update_df = cached_df.set_index('ticker')
        sector_df = sector_df.set_index('ticker')
        sector_df.update(update_df)
        return sector_df.reset_index()

    # 3. API 抓取：如果沒有快取，執行連線作業
    unknown_mask = sector_df['sector'] == 'Unknown'
    unknown_tickers = sector_df[unknown_mask]['ticker'].tolist()
    
    if not unknown_tickers:
        return sector_df

    print(f"找不到本機快取，正在透過 API 補齊 {len(unknown_tickers)} 檔股票的產業分類...")
    
    yf_logger = logging.getLogger('yfinance')
    original_level = yf_logger.level
    yf_logger.setLevel(logging.CRITICAL) 
    
    fixed_sectors = []
    
    for i, ticker in enumerate(unknown_tickers):
        try:
            info = yf.Ticker(ticker).info
            sector = info.get('sector', 'Unknown')
            fixed_sectors.append({'ticker': ticker, 'sector': sector})
            time.sleep(0.02) 
        except Exception:
            fixed_sectors.append({'ticker': ticker, 'sector': 'Unknown'})
            
        if (i + 1) % 50 == 0:
            print(f"已處理 {i + 1} / {len(unknown_tickers)}...")
            
    yf_logger.setLevel(original_level)
    
    # 4. 儲存快取：將剛抓下來的資料存成 CSV，下次就不用再抓了
    fetched_df = pd.DataFrame(fixed_sectors)
    fetched_df.to_csv(save_path, index=False)
    print(f"API 抓取完畢！已將動態產業分類永久儲存至: {save_path}")
    
    # 更新回原本的 DataFrame
    update_df = fetched_df.set_index('ticker')
    sector_df = sector_df.set_index('ticker')
    sector_df.update(update_df)
    sector_df = sector_df.reset_index()
    
    remaining = len(sector_df[sector_df['sector'] == 'Unknown'])
    print(f"補齊完成！剩餘真實無法識別(已下市)的股票數量: {remaining}")
    
    return sector_df


def preprocess_prices(prices_df, min_days=MIN_HISTORY_DAYS):
    """
    數據預處理：樞紐、前向填充、去除稀疏股票
    
    檢核清單項：
    ✓ 缺失值填充（最多 5 天前向填充）
    ✓ 去除歷史資料不足的股票
    ✓ 對齊時間序列
    """
    # 樞紐表：時間 × 股票
    pivot = prices_df.pivot_table(index='date', columns='ticker', values='close', aggfunc='last')
    pivot.index = pd.to_datetime(pivot.index)
    pivot.sort_index(inplace=True)
    
    # 前向填充（最多 5 天）
    pivot.ffill(limit=5, inplace=True)
    
    # 保留有足夠歷史資料的股票
    valid = pivot.columns[pivot.notna().sum() >= min_days]
    pivot = pivot[valid]
    
    print(f'✓ 數據矩陣：{len(pivot)} 天 × {len(pivot.columns)} 檔股票')
    print(f'✓ 時間範圍：{pivot.index[0].date()} ~ {pivot.index[-1].date()}')
    
    return pivot


# === 加載和預處理數據 ===
print("\n從資料庫加載原始數據...")
prices_raw, sector_info = load_data_from_db(DB_PATH, START_DATE, END_DATE)

print("\n預處理價格數據...")
price_pivot = preprocess_prices(prices_raw, min_days=MIN_HISTORY_DAYS)

# 建立行業對應字典
sector_map = sector_info.set_index('ticker')['sector'].to_dict()
print(f'\n【行業分佈】')
print(pd.Series(sector_map).value_counts().head(10))


從資料庫加載原始數據...
✓ 加載價格數據：3,621,962 筆記錄
✓ 加載行業數據：502 檔股票

預處理價格數據...
✓ 數據矩陣：6539 天 × 665 檔股票
✓ 時間範圍：2000-01-03 ~ 2025-12-31

【行業分佈】
Industrials               79
Financials                76
Information Technology    73
Health Care               58
Consumer Discretionary    48
Consumer Staples          35
Utilities                 31
Real Estate               31
Materials                 26
Communication Services    23
Name: count, dtype: int64


In [4]:
# 板塊過濾（如果設定了 TARGET_SECTOR）
if TARGET_SECTOR is not None:
    # 篩選出符合目標產業的股票 ticker
    target_tickers = sector_info[sector_info['sector'] == TARGET_SECTOR]['ticker'].tolist()
    print(f"🎯 板塊過濾：篩選 {TARGET_SECTOR} 板塊之股票，共 {len(target_tickers)} 檔")
    
    # 過濾 prices_raw：只保留目標產業的股票
    prices_raw = prices_raw[prices_raw['ticker'].isin(target_tickers)]
    
    # 過濾 sector_info：只保留目標產業的資訊
    sector_info = sector_info[sector_info['sector'] == TARGET_SECTOR]
else:
    print("🌍 全市場模式：使用全部 S&P 500 股票")

# ============================================================

# 
price_pivot= preprocess_prices(prices_raw)
sector_map = sector_info.set_index('ticker')['sector'].to_dict()

🌍 全市場模式：使用全部 S&P 500 股票
✓ 數據矩陣：6539 天 × 665 檔股票
✓ 時間範圍：2000-01-03 ~ 2025-12-31


In [5]:
# === 協整與參數計算 Johansen 檢定 ===
def perform_johansen_selection(form_prices, top_n=TOP_N_PAIRS, transaction_cost=TRANSACTION_COST):
    """
    學術增強版：
    1. 10% 形成期期末報酬差距預過濾。
    2. Johansen Trace Stat > 95% 臨界值。
    3. 保障基礎利潤：Spread 波動度(sigma) 必須足以完整覆蓋來回與對沖手續費。
    """
    import statsmodels.api as sm
    from statsmodels.tsa.vector_ar.vecm import coint_johansen
    import pandas as pd
    
    form_prices = form_prices.dropna(axis=1)
    if form_prices.shape[1] < 2: return []
    
    # 歸一化 (尾盤價格/初始價格 = 1.0)
    norm_p = (form_prices / form_prices.iloc[0]).dropna(axis=1)
    cols = norm_p.columns.tolist()
    
    # 預過濾條件 1：期末報酬率總差值不得超過 10%
    returns_diff = norm_p.iloc[-1] - 1
    
    results = []
    
    # 快速過濾具有方向共同性趨勢的標的
    corr_matrix = norm_p.corr()
    
    pairs = []
    for i in range(len(cols)):
        for j in range(i+1, len(cols)):
            if abs(returns_diff[cols[i]] - returns_diff[cols[j]]) <= 0.10:
                if corr_matrix.iloc[i, j] > 0.5:  
                    pairs.append((cols[i], cols[j]))

    for c1, c2 in pairs:
        y = norm_p[[c1, c2]].values
        try:
            res = coint_johansen(y, det_order=0, k_ar_diff=1)
            trace_stat = res.lr1[0]
            trace_crit = res.cvt[0, 1]  # 95% 臨界值
            if trace_stat > trace_crit: 
                # 計算組合參數: Spread = P1 - beta * P2
                P1 = norm_p[c1]
                P2 = norm_p[c2]
                P2_const = sm.add_constant(P2)
                ols_res = sm.OLS(P1, P2_const).fit()
                
                # 判斷是否回傳常數項
                beta = ols_res.params[c2] if c2 in ols_res.params else ols_res.params.iloc[-1]
                
                if abs(beta) > 3.0 or abs(beta) < 0.33 or beta < 0:
                    continue
                
                spread = P1 - beta * P2
                mu = spread.mean()
                sigma = spread.std()
                
                # 預過濾條件 2：價格波動度必須夠大，確保 2.0 Sigma 的利潤空間扣掉雙邊手續費與部位數量乘數後還有賺
                if sigma <= (1 + abs(beta)) * transaction_cost:
                    continue
                
                results.append({
                    'pair': (c1, c2),
                    'trace_stat': trace_stat,
                    'beta': beta,
                    'mu': mu,
                    'sigma': sigma,

                    'p1_0': form_prices[c1].iloc[0],
                    'p2_0': form_prices[c2].iloc[0]
                })
        except Exception as e:
            # print(f"Johansen Error {c1}-{c2}: {e}")
            pass
            
    df_res = pd.DataFrame(results)
    if df_res.empty: return []
    # 根據高波動潛力篩選：既然已經滿足 95% 協整臨界值，則優先挑選 Spread 波動空間(sigma)最大的前 N 對
    df_res = df_res.sort_values('sigma', ascending=False).head(top_n)
    
    return df_res.to_dict('records')

In [6]:
# === 交易邏輯與資本狀態機 ===
def run_trading_period(trade_prices, bench_series, config_pairs, initial_sub_capital=INITIAL_CAPITAL/NUM_SUBPORTFOLIOS):
    """
    一、 正規化：交易期開始首日再次縮放為 1
    二、 Beta 對沖：依據 Beta 調配 Long/Short 曝險量
    三、 Market Fill：剩餘與提早平倉資金全額投入 SPY 領取市場報酬
    """
    dates = trade_prices.index
    valid_pairs_count = len(config_pairs)
    slots = max(TOP_N_PAIRS, valid_pairs_count)
    capital_per_slot = initial_sub_capital / slots
    
    market_fill_capital = 0.0
    if valid_pairs_count < MIN_VALID_PAIRS:
        market_fill_capital = (slots - valid_pairs_count) * capital_per_slot
        
    portfolio_value = pd.Series(0.0, index=dates)
    trades_list = []
    
    # 1. 處理純 Market Fill (例如完全找不到配對時的情境)
    if market_fill_capital > 0:
        b0 = bench_series.iloc[0]
        spy_shares = (market_fill_capital * (1 - TRANSACTION_COST)) / b0
        portfolio_value += bench_series * spy_shares
    elif valid_pairs_count == 0:
        b0 = bench_series.iloc[0]
        spy_shares = (initial_sub_capital * (1 - TRANSACTION_COST)) / b0
        portfolio_value += bench_series * spy_shares
        return {'pnl': portfolio_value, 'trades': trades_list}

    # 2. 核心配對交易迴圈
    for cfg in config_pairs:
        c1, c2 = cfg['pair']
        if c1 not in trade_prices.columns or c2 not in trade_prices.columns:
            # 無法配對時退回 SPY
            b0 = bench_series.iloc[0]
            spy_sh = (capital_per_slot * (1 - TRANSACTION_COST)) / b0
            portfolio_value += bench_series * spy_sh
            continue
            
        p1 = trade_prices[c1]
        p2 = trade_prices[c2]
        
        # 交易期歸一化基準
        p1_0 = cfg['p1_0']
        p2_0 = cfg['p2_0']
        
        if p1.isna().any() or p2.isna().any() or pd.isna(p1_0) or pd.isna(p2_0):
            # 如果交易期內有任何一天缺少報價 (下市、停牌)，直接放棄配對，資金轉入 Market Fill
            b0 = bench_series.iloc[0]
            spy_sh = (capital_per_slot * (1 - TRANSACTION_COST)) / b0
            portfolio_value += bench_series * spy_sh
            continue

        if p1_0 == 0 or p2_0 == 0 or pd.isna(p1_0) or pd.isna(p2_0):
            b0 = bench_series.iloc[0]
            spy_sh = (capital_per_slot * (1 - TRANSACTION_COST)) / b0
            portfolio_value += bench_series * spy_sh
            continue
            
        norm_p1 = p1 / p1_0
        norm_p2 = p2 / p2_0
        
        beta, mu, sigma = cfg['beta'], cfg['mu'], cfg['sigma']
        spread = norm_p1 - beta * norm_p2
        
        position = 0 # 0:空 , 1:做多1做空2, -1:做空1做多2
        shares1_norm = 0
        shares2_norm = 0
        pair_value_daily = np.zeros(len(dates))
        
        # 資金初始投放在 SPY (Market Fill) 中等待配對建倉信號
        in_market_fill = True
        is_bankrupt = False
        current_pair_cash = 0
        fill_shares = (capital_per_slot * (1 - TRANSACTION_COST)) / bench_series.iloc[0]
        
        threshold_up = mu + Z_ENTRY * sigma
        threshold_dn = mu - Z_ENTRY * sigma
        
        active_trade = None
        
        for t in range(len(dates)):
            if is_bankrupt:
                pair_value_daily[t] = 0
                continue
            exit_signal = False
            exit_reason = ""

            date_t = dates[t]
            val1, val2 = p1.iloc[t], p2.iloc[t]
            n_val1, n_val2 = norm_p1.iloc[t], norm_p2.iloc[t]
            sp = spread.iloc[t]
            b_val = bench_series.iloc[t]
            
            # 每日結算放空保證金利息與借券費 (僅計放空腿)
            if position != 0:
                short_fee = 0
                if shares1_norm < 0:
                    short_fee += abs(shares1_norm * n_val1) * (SHORT_FEE_ANNUAL / 252)
                if shares2_norm < 0:
                    short_fee += abs(shares2_norm * n_val2) * (SHORT_FEE_ANNUAL / 252)
                current_pair_cash -= short_fee

                current_pair_value = current_pair_cash + (shares1_norm * n_val1 + shares2_norm * n_val2)
                if current_pair_value < (prev_cash_total * 0.2): # 跌掉80%強制斷頭
                    exit_signal = True
                    exit_reason = "Margin Call"
                    print("爆倉")
                
            if position == 0:
                if sp > threshold_up:
                    position = -1 # Spread 高估: 賣出(放空) 1，買入 2
                elif sp < threshold_dn:
                    position = 1  # Spread 低估: 買入 1，賣出(放空) 2
                    
                if position != 0: 
                    # 觸發建倉信號：賣出所有閒置 Market Fill 倉位轉換成現金
                    if in_market_fill:
                        current_pair_cash = fill_shares * b_val * (1 - TRANSACTION_COST)
                        in_market_fill = False
                        fill_shares = 0
                        
                    # 在計算 Q 之前加入防呆與破產機制
                    if current_pair_cash <= 0:
                        # 已經破產，資金歸零，不再建倉
                        print("已經破產了")
                        in_market_fill = False
                        current_pair_cash = 0
                        position = 0
                        is_bankrupt = True
                        continue
                    
                    # Beta 對沖數量計算
                    Q = current_pair_cash / (1 + abs(beta))
                    shares1_norm = position * Q / n_val1
                    shares2_norm = -position * beta * Q / n_val2
                        
                    trade_cost = (abs(shares1_norm * n_val1) + abs(shares2_norm * n_val2)) * TRANSACTION_COST 
                    prev_cash_total = current_pair_cash
                    
                    # 買多支出現金，放空獲得現金，並扣除手續費
                    current_pair_cash -= (shares1_norm * n_val1 + shares2_norm * n_val2 + trade_cost)
                    
                    active_trade = {
                        'stock_a': c1, 'stock_b': c2, 
                        'entry_date': date_t, 'entry_spread': sp,
                        'position': position, 'entry_cash': prev_cash_total
                    }
                    
            else:
                # 檢查平倉
                if not exit_signal: 
                    if position == -1 and sp <= mu:
                        exit_signal = True
                        exit_reason = "Mean Reversion"
                    elif position == 1 and sp >= mu:
                        exit_signal = True
                        exit_reason = "Mean Reversion"
                    elif t == len(dates) - 1:
                        exit_signal = True 
                        exit_reason = "End of Window"
                if position == -1 and sp <= mu: 
                    exit_signal = True
                    exit_reason = "Mean Reversion"
                elif position == 1 and sp >= mu: 
                    exit_signal = True
                    exit_reason = "Mean Reversion"
                elif t == len(dates) - 1: 
                    exit_signal = True # 交易視窗結束強制平倉
                    exit_reason = "End of Window"
                
                if exit_signal:
                    val_1_now = shares1_norm * n_val1
                    val_2_now = shares2_norm * n_val2
                    position_gross = abs(val_1_now) + abs(val_2_now)
                    cost = position_gross * TRANSACTION_COST
                    
                    # 平多獲得現金，平空支出現金，再扣費
                    current_pair_cash += (val_1_now + val_2_now - cost)
                    
                    if active_trade is not None:
                        active_trade['exit_date'] = date_t
                        active_trade['exit_spread'] = sp
                        active_trade['exit_reason'] = exit_reason
                        active_trade['profit'] = current_pair_cash - active_trade['entry_cash']
                        active_trade['hold_days'] = (date_t - active_trade['entry_date']).days
                        trades_list.append(active_trade)
                        active_trade = None
                        
                    position, shares1_norm, shares2_norm = 0, 0, 0
                    
                    # 平倉後立刻再次投入 Market Fill 吃到期末
                    if t < len(dates) - 1:
                        if pd.isna(current_pair_cash) or current_pair_cash <= 0:
                            # 徹底破產或發生 NaN 異常，該槽位資金歸零，直接終止
                            fill_shares = 0
                            in_market_fill = False
                            current_pair_cash = 0 
                            is_bankrupt = True
                        else:
                            # 只有在資金為正時，才能繼續買入大盤 Market Fill
                            fill_shares = (current_pair_cash * (1 - TRANSACTION_COST)) / b_val
                            in_market_fill = True
        is_bankrupt = False
                            current_pair_cash = 0
                    
            if in_market_fill:
                 pair_value_daily[t] = fill_shares * b_val
            else:
                 pair_value_daily[t] = current_pair_cash + (shares1_norm * n_val1) + (shares2_norm * n_val2)
                
        portfolio_value += pair_value_daily
        
    return {'pnl': portfolio_value, 'trades': trades_list}

## 階段 4：滾動視窗回測框架

### 檢核清單：
- ✓ **形成期**：252 天（約 1 年）
- ✓ **交易期**：126 天（約 6 個月）
- ✓ **滾動步長**：20 天（約 1 個月）
- ✓ **梯隊資金**：將資本分割到每個滾動視窗
- ✓ **聚合損益**：逐日累積所有視窗的損益

In [ ]:
# ========== 階段 4：滾動視窗回測框架 ==========
def run_coin_backtest(price_pivot, sector_map,
                     formation_window=FORMATION_WINDOW,
                     trading_window=TRADING_WINDOW,
                     rolling_window=ROLLING_WINDOW,
                     top_n=TOP_N_PAIRS,
                     initial_capital=INITIAL_CAPITAL):
    """
    完整的滾動視窗 SSD 配對交易回測 - 修正槽位計算法
    """
    
    dates = price_pivot.index
    N = len(dates)
    
    # 持續投資的時間槽(Slots)，每個槽位一份資金
    num_subportfolios = math.ceil(trading_window / rolling_window)
    tranche_capital = initial_capital / num_subportfolios
    
    # 紀錄各個槽位每天的絕對價值，初始預設為現金 tranche_capital
    slots_daily_value = np.full((num_subportfolios, N), tranche_capital)
    
    all_trades = []
    window_records = []
    
    print(f"\n【回測配置】")
    print(f"  資本槽位數: {num_subportfolios}")
    print(f"  每槽位配置: ${tranche_capital:,.2f}")
    print(f"  時間跨度: {dates[0].date()} ~ {dates[-1].date()}")
    
    # ===== 滾動視窗迴圈 =====
    si, window_id = 0, 0
    
    while si + formation_window + trading_window <= N:
        fe = si + formation_window        # 形成期結束
        te = fe + trading_window          # 交易期結束
        
        form_prices = price_pivot.iloc[si:fe]
        trade_prices = price_pivot.iloc[fe:te]
        
        # 排除形成期最後一天股價低於 $2 的股票
        valid_tickers = form_prices.columns[form_prices.iloc[-1] >= 2.0]
        form_prices = form_prices[valid_tickers]

        # 配對篩選
        pairs = perform_johansen_selection(form_prices, top_n=top_n)
        
        slot_idx = window_id % num_subportfolios
        # 利用此槽位在交易期第一天(fe)的現有可動用資金進行交易
        current_slot_cap = slots_daily_value[slot_idx, fe]
        
        # 決定 Bench_series (若無 SPY 則採平均報酬作為等權重替代品)
        if 'SPY' in trade_prices.columns:
            bench_prices = trade_prices['SPY']
        else:
            bench_prices = trade_prices.mean(axis=1)
            bench_prices = bench_prices / bench_prices.iloc[0] * current_slot_cap
            
        # 本視窗的槽位資金全部丟入單次 run_trading_period
        result = run_trading_period(trade_prices, bench_prices, pairs, current_slot_cap)        
        window_pnl = result['pnl']
        window_all_trades = result['trades']
        
        # 覆蓋該槽位在此交易期間的每日實際價值
        slots_daily_value[slot_idx, fe:te] = window_pnl.values
        # 交易期結束後將最新剩餘資金平移為現金，直到該槽位再次被重新配對
        slots_daily_value[slot_idx, te:] = window_pnl.iloc[-1]
        
        all_trades.extend(window_all_trades)
        
        # 計算本視窗績效
        if current_slot_cap <= 0:
            window_return = -1.0  # 本金已歸零或為負，該視窗報酬率強制記為 -100%
        else:
            window_return = (window_pnl.iloc[-1] - current_slot_cap) / current_slot_cap
        trade_count = len(window_all_trades)
        
        window_records.append({
            'window_id': window_id,
            'slot_id': slot_idx,
            'form_start': dates[si],
            'form_end': dates[fe-1],
            'trade_start': dates[fe],
            'trade_end': dates[te-1],
            'pairs_count': len(pairs),
            'trade_count': trade_count,
            'return_pct': window_return * 100,
            'pnl_total': window_pnl.iloc[-1] - current_slot_cap
        })
        
        # 列印進度
        print(f"  W{window_id:03d} (Slot {slot_idx}) | {dates[fe].date()} ~ {dates[te-1].date()} | "
                  f"配對: {len(pairs)} | 交易: {trade_count} | 報酬: {window_return*100:+.2f}%")
        
        si += rolling_window
        window_id += 1
    
    # 總淨值為所有槽位價值加總
    total_portfolio_nav = slots_daily_value.sum(axis=0)
    all_pnl = pd.DataFrame({'Total_NAV': total_portfolio_nav}, index=dates)
    
    print(f"\n✓ 回測完成：{window_id} 個視窗，{len(all_trades)} 筆交易")
    
    return {
        'pnl': all_pnl,
        'trades': all_trades,
        'windows': window_records
    }

# ===== 執行回測 =====
print("\n執行滾動視窗回測...")
backtest_result = run_coin_backtest(price_pivot, sector_map)


執行滾動視窗回測...

【回測配置】
  資本槽位數: 6
  每槽位配置: $1,666.67
  時間跨度: 2000-01-03 ~ 2025-12-31
  W000 (Slot 0) | 2001-01-02 ~ 2001-07-02 | 配對: 20 | 交易: 16 | 報酬: +0.12%
  W001 (Slot 1) | 2001-02-01 ~ 2001-08-01 | 配對: 20 | 交易: 16 | 報酬: -14.89%
  W002 (Slot 2) | 2001-03-05 ~ 2001-08-30 | 配對: 20 | 交易: 8 | 報酬: -13.95%
  W003 (Slot 3) | 2001-04-03 ~ 2001-10-05 | 配對: 20 | 交易: 22 | 報酬: +3.16%
  W004 (Slot 4) | 2001-05-03 ~ 2001-11-05 | 配對: 20 | 交易: 15 | 報酬: -17.02%
  W005 (Slot 5) | 2001-06-04 ~ 2001-12-05 | 配對: 20 | 交易: 18 | 報酬: -15.88%
  W006 (Slot 0) | 2001-07-03 ~ 2002-01-07 | 配對: 20 | 交易: 15 | 報酬: +1.43%
  W007 (Slot 1) | 2001-08-02 ~ 2002-02-06 | 配對: 20 | 交易: 12 | 報酬: -5.78%
  W008 (Slot 2) | 2001-08-31 ~ 2002-03-08 | 配對: 20 | 交易: 12 | 報酬: +7.77%
  W009 (Slot 3) | 2001-10-08 ~ 2002-04-09 | 配對: 20 | 交易: 21 | 報酬: +17.52%
  W010 (Slot 4) | 2001-11-06 ~ 2002-05-08 | 配對: 20 | 交易: 17 | 報酬: +10.81%
  W011 (Slot 5) | 2001-12-06 ~ 2002-06-07 | 配對: 20 | 交易: 16 | 報酬: -2.35%
爆倉
  W012 (Slot 0) | 2002-01-08 ~ 20

## 階段 5：績效指標計算與分析

### 計算指標：
- **CAGR (年化報酬率)**：$\text{CAGR} = (1 + R)^{252/n} - 1$
- **年化波動率**：$\sigma_{annual} = \sigma_{daily} \times \sqrt{252}$
- **Sharpe Ratio**：$SR = \frac{r_{annual}}{\sigma_{annual}}$
- **Sortino Ratio**：$\text{Sortino} = \frac{r_{annual}}{\sigma_{downside}}$
- **最大回撤 (MDD)**：$\text{MDD} = \min\left(\frac{V_t - V_{max}}{V_{max}}\right)$
- **勝率**：$\frac{\text{正報酬日數}}{\text{總交易日數}}$
- **平均獲利/虧損**：配對層級統計

In [8]:
# ========== 階段 5：績效指標計算與分析 ==========
def compute_performance_metrics(pnl_df, initial_capital, all_trades):
    """計算完整的績效指標"""
    
    # ===== 修正報酬率算法 =====
    # pnl_df 已經包含每日的總 NAV，計算單日報酬率與總淨值
    portfolio_pnl = pnl_df['Total_NAV']
    returns = portfolio_pnl.pct_change().dropna()
    
    if len(returns) == 0:
        print("⚠️  無有效報酬數據")
        return None
    
    # ===== 淨值曲線 =====
    # nav 已經是實際總資金比例
    nav = portfolio_pnl / initial_capital
    
    # ===== 基本指標 =====
    total_return = nav.iloc[-1] - 1
    cagr = (1 + total_return) ** (252 / len(returns)) - 1 if total_return > -1 else -1
    annual_vol = returns.std() * np.sqrt(252)
    
    # ===== 比率指標 =====
    sharpe = (returns.mean() * 252) / annual_vol if annual_vol > 1e-8 else np.nan
    
    # Sortino Ratio（僅計算下行波動率）
    downside_returns = returns[returns < 0]
    downside_vol = downside_returns.std() * np.sqrt(252) if len(downside_returns) > 0 else 0
    sortino = (returns.mean() * 252) / downside_vol if downside_vol > 1e-8 else np.nan
    
    # ===== 風險指標 =====
    drawdown = (nav - nav.cummax()) / nav.cummax()
    mdd = drawdown.min()
    
    # 恢復時間
    max_dd_date = drawdown.idxmin()
    recovery_dates = nav[nav.index > max_dd_date][nav >= nav[nav.index <= max_dd_date].max()]
    recovery_days = (recovery_dates.index[0] - max_dd_date).days if len(recovery_dates) > 0 else np.nan
    
    # ===== 交易統計 =====
    win_rate = (returns > 0).sum() / len(returns)
    
    # 配對層級統計
    if all_trades:
        trades_df = pd.DataFrame(all_trades)
        avg_profit = trades_df['profit'].mean()
        avg_hold = trades_df['hold_days'].mean()
        win_trades = (trades_df['profit'] > 0).sum()
        total_trades = len(trades_df)
        trade_win_rate = win_trades / total_trades if total_trades > 0 else 0
        avg_profit_win = trades_df[trades_df['profit'] > 0]['profit'].mean() if win_trades > 0 else 0
        avg_profit_loss = trades_df[trades_df['profit'] <= 0]['profit'].mean() if total_trades - win_trades > 0 else 0
    else:
        avg_profit = avg_hold = win_trades = total_trades = trade_win_rate = 0
        avg_profit_win = avg_profit_loss = 0.0
        
    metrics = {
        'CAGR(%)': round(cagr * 100, 2),
        '總報酬(%)': round(total_return * 100, 2),
        '年化波動(%)': round(annual_vol * 100, 2),
        'Sharpe': round(sharpe, 4),
        'Sortino': round(sortino, 4),
        '最大回撤(%)': round(mdd * 100, 2),
        '恢復天數': round(recovery_days, 0) if pd.notna(recovery_days) else float('nan'),
        '日勝率(%)': round(win_rate * 100, 2),
        '配對勝率(%)': round(trade_win_rate * 100, 2),
        '平均配對利潤': round(avg_profit, 2),
        '平均獲利配對': round(avg_profit_win, 2),
        '平均虧損配對': round(avg_profit_loss, 2),
        '平均持倉天數': round(avg_hold, 1),
        '總配對數': total_trades
    }
    
    return metrics

# ===== 計算績效 =====
print("\n計算績效指標...")
pnl_df = backtest_result['pnl']
trades = backtest_result['trades']

metrics = compute_performance_metrics(pnl_df, INITIAL_CAPITAL, trades)

if metrics:
    print("\n【核心績效指標】")
    print(f"  CAGR: {metrics['CAGR(%)']:.2f}%")
    print(f"  年化波動: {metrics['年化波動(%)']:.2f}%")
    print(f"  Sharpe: {metrics['Sharpe']:.4f}")
    print(f"  最大回撤: {metrics['最大回撤(%)']:.2f}%")
    print(f"  日勝率: {metrics['日勝率(%)']:.2f}%")
    print(f"  配對勝率: {metrics['配對勝率(%)']:.2f}%")
    print(f"  總配對交易: {metrics['總配對數']}")
    
    # 詳細指標表
    metrics_df = pd.DataFrame([metrics])
    display(metrics_df.style.format("{:.2f}", na_rep="N/A"))


計算績效指標...

【核心績效指標】
  CAGR: 299502.61%
  年化波動: 246067.39%
  Sharpe: 0.9706
  最大回撤: -99.96%
  日勝率: 46.99%
  配對勝率: 54.24%
  總配對交易: 5570


,CAGR(%),總報酬(%),年化波動(%),Sharpe,Sortino,最大回撤(%),恢復天數,日勝率(%),配對勝率(%),平均配對利潤,平均獲利配對,平均虧損配對,平均持倉天數,總配對數
0,299502.61,157414236316291689309279485986186496749189912103461908098930486489140917608168196791796236288.00,246067.39,0.97,3141.38,-99.96,2.00,46.99,54.24,2110224871414458657186120532190717939637077884824388951281640268863490935523805401981124608.00,10444135043080312346725823026308175103306072046222687198129219702809884366188714253929676800.00,-7766880906774063488891610072052886678769580805592084647730516835250165624965556728372396032.00,96.00,5570.00


## 階段 6：結果視覺化與對標

### 視覺化內容：
1. **淨值曲線 (NAV)** - 累積報酬走勢
2. **回撤曲線** - 最大回撤動態
3. **日度報酬分佈** - 報酬率直方圖
4. **月度績效熱力圖** - 時間分解
5. **配對價差時序** - 交易信號驗證
6. **交易日誌表** - 詳細交易記錄
7. **對標比較** - vs. S&P 500

In [9]:
import yfinance as yf
import pandas as pd

def fetch_benchmark_returns(target_index, ticker="^GSPC"):
    """
    Fetch benchmark daily returns and align with the strategy's trading days.
    """
    # Extract start and end dates from the strategy's index
    start_date = target_index.min().strftime('%Y-%m-%d')
    # Add one day to end_date to ensure the last day is included in yfinance
    end_date = (target_index.max() + pd.Timedelta(days=1)).strftime('%Y-%m-%d')
    
    print(f"Downloading benchmark data ({ticker}) from {start_date} to {end_date}...")
    
    # Download daily data
    df = yf.download(ticker, start=start_date, end=end_date, progress=False)
    
    if df.empty:
        print("Warning: Failed to fetch benchmark data. Returning zeros.")
        return pd.Series(0, index=target_index)
        
    # Calculate daily returns based on Adjusted Close
    if isinstance(df.columns, pd.MultiIndex):
        if 'Adj Close' in df.columns.levels[0] or 'Adj Close' in df.columns:
            try:
                close_prices = df['Adj Close'][ticker]
            except:
                close_prices = df['Close'][ticker]
        else:
            close_prices = df['Close'][ticker]
    else:
        if 'Adj Close' in df.columns:
            close_prices = df['Adj Close']
        elif 'Adj_Close' in df.columns:
            close_prices = df['Adj_Close']
        else:
            close_prices = df['Close']
            
    bench_returns = close_prices.pct_change().dropna()
    
    # Align benchmark returns with the strategy's datetime index
    aligned_returns = bench_returns.reindex(target_index).fillna(0)
    
    return aligned_returns

# ==========================================
# Integration with Step 6.1
# ==========================================

# 1. Fetch the benchmark returns using your strategy's date index
try:
    benchmark_returns = fetch_benchmark_returns(pnl_df.index, ticker="^GSPC")
    # 2. Calculate the Cumulative Net Asset Value (NAV) for the benchmark
    benchmark_nav = (1 + benchmark_returns).cumprod()
    print("✓ Benchmark data successfully loaded and aligned.")
except Exception as e:
    print(f"Warning: error loading benchmark - {e}")
    benchmark_nav = pd.Series(1, index=pnl_df.index)

✓ Benchmark data successfully loaded and aligned.


In [10]:
# ========== 階段 6：結果視覺化與對標 ==========
# ===== 準備數據 =====
portfolio_pnl = pnl_df['Total_NAV']
returns = portfolio_pnl.pct_change().dropna()
nav = portfolio_pnl / INITIAL_CAPITAL
drawdown = (nav - nav.cummax()) / nav.cummax() * 100

# ===== 1. 淨值曲線與回撤 =====
print("\n【第 6.1 步】繪製淨值曲線與回撤 (包含大盤對標)...")

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.65, 0.35],
    subplot_titles=['淨值曲線 (NAV) 比較', '最大回撤 (MDD)'],
    vertical_spacing=0.1
)

# Plot Strategy NAV
fig.add_trace(
    go.Scatter(
        x=nav.index, y=nav.values,
        name='SSD Strategy',
        line=dict(color='#1f77b4', width=2),
        hovertemplate='%{x|%Y-%m-%d}<br>Strategy NAV: %{y:.4f}<extra></extra>'
    ),
    row=1, col=1
)

# Plot Benchmark NAV 
fig.add_trace(
    go.Scatter(
        x=benchmark_nav.index, y=benchmark_nav.values,
        name='Market Benchmark',
        line=dict(color='#ff7f0e', width=2, dash='dot'),
        hovertemplate='%{x|%Y-%m-%d}<br>Benchmark NAV: %{y:.4f}<extra></extra>'
    ),
    row=1, col=1
)

# 回撤
fig.add_trace(
    go.Scatter(
        x=drawdown.index, y=drawdown.values,
        name='MDD',
        fill='tozeroy',
        line=dict(color='#d62728', width=1),
        hovertemplate='%{x|%Y-%m-%d}<br>回撤: %{y:.2f}%<extra></extra>'
    ),
    row=2, col=1
)

fig.update_layout(
    title='SSD 配對交易系統 - 淨值與回撤曲線 (2000-2025)',
    height=700,
    template='plotly_dark',
    hovermode='x unified',
    showlegend=True
)

fig.update_yaxes(title_text='NAV', row=1, col=1)
fig.update_yaxes(title_text='回撤 (%)', row=2, col=1)
fig.show()

print("✓ 淨值曲線已繪製")

# ===== 2. 日度報酬分佈 =====
print("\n【第 6.2 步】繪製報酬分佈...")

fig_returns = go.Figure()

fig_returns.add_trace(go.Histogram(
    x=returns * 100,
    name='日度報酬',
    nbinsx=50,
    marker=dict(color='#2ca02c', opacity=0.7),
    hovertemplate='報酬率: %{x:.2f}%<br>頻次: %{y}<extra></extra>'
))

fig_returns.add_vline(
    x=returns.mean() * 100,
    line_dash='dash',
    line_color='red',
    annotation_text=f"平均: {returns.mean()*100:.3f}%",
    annotation_position="top right"
)

fig_returns.update_layout(
    title='日度報酬率分佈',
    xaxis_title='報酬率 (%)',
    yaxis_title='頻次',
    height=400,
    template='plotly_dark'
)

fig_returns.show()

print("✓ 報酬分佈已繪製")

# ===== 3. 月度績效熱力圖 =====
print("\n【第 6.3 步】生成月度績效熱力圖...")

# 按月計算報酬
returns_monthly = returns.resample('M').apply(lambda x: (1 + x).prod() - 1)
returns_monthly.index = returns_monthly.index.to_period('M')

# 轉換為年-月矩陣
if len(returns_monthly) > 0:
    pivot_returns = pd.DataFrame({
        'year': [d.year for d in returns_monthly.index],
        'month': [d.month for d in returns_monthly.index],
        'return': returns_monthly.values
    })
    
    pivot_table = pivot_returns.pivot(index='year', columns='month', values='return') * 100
    
    fig_heatmap = go.Figure(data=go.Heatmap(
        z=pivot_table.values,
        x=['1月', '2月', '3月', '4月', '5月', '6月', 
           '7月', '8月', '9月', '10月', '11月', '12月'],
        y=pivot_table.index,
        colorscale='RdYlGn',
        zmid=0,
        hovertemplate='%{y}年 %{x}: %{z:.2f}%<extra></extra>'
    ))
    
    fig_heatmap.update_layout(
        title='月度報酬率熱力圖 (%)',
        xaxis_title='月份',
        yaxis_title='年份',
        height=500,
        template='plotly_dark'
    )
    
    fig_heatmap.show()
    print("✓ 熱力圖已繪製")

# ===== 4. 視窗績效統計 =====
print("\n【第 6.4 步】視窗績效統計...")

windows_df = pd.DataFrame(backtest_result['windows'])
if len(windows_df) > 0:
    print(f"\n【視窗績效摘要】(共 {len(windows_df)} 個視窗)")
    print(windows_df[['window_id', 'pairs_count', 'trade_count', 'return_pct']].head(10)
                    .to_string(index=False))
    
    print(f"\n  平均配對數: {windows_df['pairs_count'].mean():.1f}")
    print(f"  平均交易數: {windows_df['trade_count'].mean():.1f}")
    print(f"  平均視窗報酬: {windows_df['return_pct'].mean():+.2f}%")
    print(f"  正報酬視窗: {(windows_df['return_pct'] > 0).sum()}/{len(windows_df)}")

# ===== 5. 交易日誌 =====
print("\n【第 6.5 步】交易日誌統計...")

if trades:
    trades_df = pd.DataFrame(trades)
    trades_df['entry_date'] = pd.to_datetime(trades_df['entry_date'])
    trades_df['exit_date'] = pd.to_datetime(trades_df['exit_date'])
    
    print(f"\n【交易統計】(共 {len(trades_df)} 筆交易)")
    print(f"  獲利交易: {(trades_df['profit'] > 0).sum()}")
    print(f"  虧損交易: {(trades_df['profit'] <= 0).sum()}")
    print(f"  平均利潤: {trades_df['profit'].mean():.6f}")
    print(f"  最大利潤: {trades_df['profit'].max():.6f}")
    print(f"  最大虧損: {trades_df['profit'].min():.6f}")
    print(f"  平均持倉: {trades_df['hold_days'].mean():.1f} 天")
    
    # 按平倉原因分類
    print(f"\n【平倉原因統計】")
    exit_reasons = trades_df['exit_reason'].value_counts()
    for reason, count in exit_reasons.items():
        pct = count / len(trades_df) * 100
        print(f"  {reason}: {count} ({pct:.1f}%)")
    
    # 前 10 筆交易
    print(f"\n【前 10 筆交易記錄】")
    display(trades_df[['stock_a', 'stock_b', 'entry_date', 'exit_date', 
                       'entry_spread', 'exit_spread', 'profit', 'hold_days']].head(10)
                    .style.format({'profit': '{:.6f}', 'entry_spread': '{:.4f}', 
                                 'exit_spread': '{:.4f}'}))

print("\n✓ 視覺化與分析完成")


【第 6.1 步】繪製淨值曲線與回撤 (包含大盤對標)...


✓ 淨值曲線已繪製

【第 6.2 步】繪製報酬分佈...


✓ 報酬分佈已繪製

【第 6.3 步】生成月度績效熱力圖...


✓ 熱力圖已繪製

【第 6.4 步】視窗績效統計...

【視窗績效摘要】(共 294 個視窗)
 window_id  pairs_count  trade_count  return_pct
         0           20           16      0.1233
         1           20           16    -14.8912
         2           20            8    -13.9454
         3           20           22      3.1562
         4           20           15    -17.0173
         5           20           18    -15.8816
         6           20           15      1.4317
         7           20           12     -5.7778
         8           20           12      7.7750
         9           20           21     17.5249

  平均配對數: 20.0
  平均交易數: 18.9
  平均視窗報酬: +173104831113071160066982632460976683196388513920581632.00%
  正報酬視窗: 151/294

【第 6.5 步】交易日誌統計...

【交易統計】(共 5570 筆交易)
  獲利交易: 3021
  虧損交易: 2549
  平均利潤: 2110224871414458657186120532190717939637077884824388951281640268863490935523805401981124608.000000
  最大利潤: 4315281445196899687013080042071690403296719993653694925314333127791431000369830374756671553536.000000
  最大虧損: -139

,stock_a,stock_b,entry_date,exit_date,entry_spread,exit_spread,profit,hold_days
0,CIEN,CNX,2001-03-27 00:00:00,2001-07-02 00:00:00,-1.1098,-1.0022,-0.671808,97
1,MU,TKO,2001-01-16 00:00:00,2001-03-05 00:00:00,-1.6133,-0.7788,17.394190,48
2,GLW,WMB,2001-03-02 00:00:00,2001-07-02 00:00:00,-2.2261,-1.9538,0.298969,122
3,INCY,PTC,2001-01-02 00:00:00,2001-07-02 00:00:00,-0.7664,-0.6588,2.497303,181
4,GLW,TROW,2001-04-18 00:00:00,2001-07-02 00:00:00,-1.8592,-2.0026,-6.958836,75
5,EP,SBUX,2001-01-17 00:00:00,2001-02-21 00:00:00,1.2890,-0.5240,30.854070,35
6,EP,PAYX,2001-01-17 00:00:00,2001-02-21 00:00:00,1.6435,0.0048,28.635930,35
7,TER,TXT,2001-01-02 00:00:00,2001-07-02 00:00:00,-1.9127,-2.5101,-13.535626,181
8,AMAT,TXT,2001-01-02 00:00:00,2001-07-02 00:00:00,-1.0608,-1.1964,-4.544038,181
9,KLAC,TXT,2001-01-04 00:00:00,2001-07-02 00:00:00,-1.0471,-0.6882,7.427788,179



✓ 視覺化與分析完成


### 【年淨值走勢圖】
按年份展示淨值變化，清晰呈現策略在不同年度的表現。

In [11]:
# ===== 年淨值走勢圖 =====
print("\n【第 6.2.5 步】繪製年淨值走勢圖...")

# 計算每年度的淨值
nav_annual = nav.resample('Y').last()
nav_annual.index = nav_annual.index.year

# 初始值設置
nav_annual_values = [1.0]  # 2000年初始淨值
for i in range(len(nav_annual) - 1):
    nav_annual_values.append(nav_annual.iloc[i])

nav_annual_values = nav_annual.values
years = nav_annual.index.astype(str).astype(int).tolist()

# 繪製年淨值走勢折線圖
fig_nav_annual = go.Figure()

# 添加淨值線
fig_nav_annual.add_trace(go.Scatter(
    x=years,
    y=nav_annual_values,
    mode='lines+markers',
    name='年末淨值',
    line=dict(color='#1f77b4', width=3),
    marker=dict(size=8, symbol='circle'),
    fill='tozeroy',
    fillcolor='rgba(31, 119, 180, 0.2)',
    hovertemplate='%{x}年<br>淨值: %{y:.4f}<extra></extra>'
))

# 添加年初淨值 (初始值為1)
years_with_initial = [years[0] - 1] + years
nav_values_with_initial = [1.0] + nav_annual_values.tolist()

fig_nav_annual.add_trace(go.Scatter(
    x=years_with_initial,
    y=nav_values_with_initial,
    mode='lines+markers',
    name='累積淨值',
    line=dict(color='#ff7f0e', width=2, dash='dash'),
    marker=dict(size=6),
    hovertemplate='%{x}年<br>累積淨值: %{y:.4f}<extra></extra>',
    showlegend=False
))

# 更新佈局
fig_nav_annual.update_layout(
    title='SSD 配對交易系統 - 年淨值走勢圖 (2000-2025)',
    xaxis_title='年份',
    yaxis_title='淨值',
    height=500,
    template='plotly_dark',
    hovermode='x unified',
    xaxis=dict(
        tickmode='linear',
        tick0=2000,
        dtick=1
    ),
    yaxis=dict(
        gridcolor='rgba(128, 128, 128, 0.2)'
    ),
    font=dict(size=11),
    showlegend=True,
    legend=dict(
        orientation='v',
        yanchor='top',
        y=0.99,
        xanchor='left',
        x=0.01,
        bgcolor='rgba(0, 0, 0, 0.5)'
    )
)

# 添加數值標籤
for year, nav_val in zip(years, nav_annual_values):
    fig_nav_annual.add_annotation(
        x=year,
        y=nav_val,
        text=f'{nav_val:.2f}',
        showarrow=True,
        arrowhead=2,
        arrowsize=1,
        arrowwidth=1,
        arrowcolor='#1f77b4',
        ax=0,
        ay=-20,
        font=dict(size=9, color='#1f77b4')
    )

fig_nav_annual.show()

print("✓ 年淨值走勢圖已繪製")

# 計算年度變化統計
print("\n【年度淨值統計】")
print(f"{'年份':<8} {'年末淨值':<12} {'年度報酬(%)':<12} {'累計報酬(%)':<12}")
print("-" * 46)

prev_nav = 1.0
for year, nav_val in zip(years, nav_annual_values):
    annual_return = (nav_val - prev_nav) / prev_nav * 100
    cumulative_return = (nav_val - 1) * 100
    print(f"{year:<8} {nav_val:<12.4f} {annual_return:>10.2f}% {cumulative_return:>10.2f}%")
    prev_nav = nav_val

# 最佳年份和最差年份
best_year_idx = np.argmax(nav_annual_values)
worst_year_idx = np.argmin(nav_annual_values)

print(f"\n【極值統計】")
print(f"  最佳年份: {years[best_year_idx]} (淨值: {nav_annual_values[best_year_idx]:.4f})")
print(f"  最差年份: {years[worst_year_idx]} (淨值: {nav_annual_values[worst_year_idx]:.4f})")
print(f"  年均淨值: {np.mean(nav_annual_values):.4f}")
print(f"  年末淨值: {nav_annual_values[-1]:.4f} (總增長: {(nav_annual_values[-1]-1)*100:.2f}%)")


【第 6.2.5 步】繪製年淨值走勢圖...


✓ 年淨值走勢圖已繪製

【年度淨值統計】
年份       年末淨值         年度報酬(%)      累計報酬(%)     
----------------------------------------------
2000     1.0000             0.00%       0.00%
2001     0.9464            -5.36%      -5.36%
2002     0.9486             0.24%      -5.14%
2003     0.9899             4.35%      -1.01%
2004     0.9659            -2.43%      -3.41%
2005     1.0417             7.85%       4.17%
2006     1.3114            25.89%      31.14%
2007     17887.0133   1363863.86% 1788601.33%
2008     12962.4295       -27.53% 1296142.95%
2009     16440.4380        26.83% 1643943.80%
2010     270687.4392     1546.47% 27068643.92%
2011     90514.7919       -66.56% 9051379.19%
2012     991244687337378432.0000 1095119003306196.88% 99124468733737844736.00%
2013     677717033395409664.0000     -31.63% 67771703339540963328.00%
2014     417245923280478787264312372273006183501808592884953884446858309271552.0000 61566391682683194273642649173072354846694525970153472.00% 417245923280478815998603763508422277720

## 進階分析：時間與部門分解

## 數據匯出與存檔

In [12]:
# === 匯出回測結果 ===
print("\n【數據匯出】")

# 確保 results 目錄存在
os.makedirs(r'..\results', exist_ok=True)

# 1. 導出 PnL
pnl_export = pnl_df.sum(axis=1)
pnl_export.to_csv(r'..\results\ssd_pnl.csv')
print("✓ 已匯出日度損益 -> results/ssd_pnl.csv")

# 2. 導出淨值
nav_export = (1 + (pnl_export / INITIAL_CAPITAL)).cumprod()
nav_export.to_csv(r'..\results\ssd_nav.csv')
print("✓ 已匯出淨值曲線 -> results/ssd_nav.csv")

# 3. 導出交易日誌
if trades:
    trades_df_export = pd.DataFrame(trades)
    trades_df_export.to_csv(r'..\results\ssd_trades.csv', index=False)
    print(f"✓ 已匯出交易日誌 ({len(trades_df_export)} 筆) -> results/ssd_trades.csv")

# 4. 導出視窗績效
windows_df_export = pd.DataFrame(backtest_result['windows'])
windows_df_export.to_csv(r'..\results\ssd_windows.csv', index=False)
print(f"✓ 已匯出視窗績效 ({len(windows_df_export)} 個) -> results/ssd_windows.csv")

# 5. 導出績效指標
if metrics:
    pd.DataFrame([metrics]).to_csv(r'..\results\ssd_metrics.csv', index=False)
    print("✓ 已匯出績效指標 -> results/ssd_metrics.csv")

print("\n✅ 回測完整！所有結果已保存到 results/ 目錄")


【數據匯出】
✓ 已匯出日度損益 -> results/ssd_pnl.csv
✓ 已匯出淨值曲線 -> results/ssd_nav.csv
✓ 已匯出交易日誌 (5570 筆) -> results/ssd_trades.csv
✓ 已匯出視窗績效 (294 個) -> results/ssd_windows.csv
✓ 已匯出績效指標 -> results/ssd_metrics.csv

✅ 回測完整！所有結果已保存到 results/ 目錄
